In [1]:
import numpy as np
import pandas
import pm4py
from matplotlib import pyplot as plt
from sklearn.mixture import GaussianMixture
import scipy.stats as stats
import ot
import os
from tqdm import tqdm
import collections
import matplotlib.dates as md
import importlib
import pickle
import random
import math
import CRPS.CRPS as pscore
import datetime

import multiprocessing as mp
mp.set_start_method('spawn')


np.seterr(all='warn', over='ignore')
pandas.set_option('display.max_columns', None)
#pandas.set_option('display.max_rows', None)


import sys
sys.path.append('../../TaskExecutionTimeMining/')
from drbart_parser import *
from event_log_transformer import *

#sys.path.append('../../Evaluation')
sys.path.append('../../Evaluation/')
from conduct_evaluation import ConductEvaluation
from normal_evaluation.drbart_evaluation import *

get_pscores = lambda likelihoods : [pscore(likelihoods[1][i], likelihoods[2][k][3]).compute()[0] for i, k in enumerate(list(likelihoods[0].keys()))]

In [2]:
#model_name = 'bpic_2017_all_2'
model_name = 'bpic_2017'

n_processes = 128
batch_size = 25

log_name = 'test'

with open('../transformed_event_logs/BPIC_2017_all_'+log_name+'.pickle', 'rb') as f:
    test_event_log = pickle.load(f)

test_event_log['case:concept:name'] = test_event_log['case:concept:name'].astype(str)
known_resources = ['User_1','User_10','User_100','User_101','User_102','User_103','User_104','User_105','User_106','User_107','User_108','User_109','User_11','User_110','User_111','User_112','User_113','User_114','User_115','User_116','User_117','User_118','User_119','User_12','User_120','User_121','User_122','User_123','User_124','User_125','User_126','User_127','User_128','User_129','User_13','User_130','User_131','User_132','User_133','User_134','User_135','User_136','User_137','User_138','User_139','User_14','User_140','User_141','User_142','User_143','User_144','User_145','User_146','User_147','User_148','User_149','User_15','User_16','User_17','User_18','User_19','User_2','User_20','User_21','User_22','User_23','User_24','User_25','User_26','User_27','User_28','User_29','User_3','User_30','User_31','User_32','User_33','User_34','User_35','User_36','User_37','User_38','User_39','User_4','User_40','User_41','User_42','User_43','User_44','User_45','User_46','User_47','User_48','User_49','User_5','User_50','User_51','User_52','User_53','User_54','User_55','User_56','User_57','User_58','User_59','User_6','User_60','User_61','User_62','User_63','User_64','User_65','User_66','User_67','User_68','User_69','User_7','User_70','User_71','User_72','User_73','User_74','User_75','User_76','User_77','User_78','User_79','User_8','User_80','User_81','User_82','User_83','User_84','User_85','User_86','User_87','User_88','User_89','User_9','User_90','User_91','User_92','User_93','User_94','User_95','User_96','User_97','User_98','User_99']
known_activities = ['W_Assess potential fraud__ate_abort','W_Assess potential fraud__complete','W_Assess potential fraud__resume','W_Assess potential fraud__schedule','W_Assess potential fraud__start','W_Assess potential fraud__suspend','W_Assess potential fraud__withdraw','W_Call after offers__ate_abort','W_Call after offers__complete','W_Call after offers__resume','W_Call after offers__schedule','W_Call after offers__start','W_Call after offers__suspend','W_Call after offers__withdraw','W_Call incomplete files__ate_abort','W_Call incomplete files__complete','W_Call incomplete files__resume','W_Call incomplete files__schedule','W_Call incomplete files__start','W_Call incomplete files__suspend','W_Complete application__ate_abort','W_Complete application__complete','W_Complete application__resume','W_Complete application__schedule','W_Complete application__start','W_Complete application__suspend','W_Handle leads__complete','W_Handle leads__resume','W_Handle leads__schedule','W_Handle leads__start','W_Handle leads__suspend','W_Handle leads__withdraw','W_Shortened completion __resume','W_Shortened completion __schedule','W_Shortened completion __start','W_Shortened completion __suspend','W_Validate application__ate_abort','W_Validate application__complete','W_Validate application__resume','W_Validate application__schedule','W_Validate application__start','W_Validate application__suspend']
ii1 = ['intercase_n_1__W_Assess potential fraud__ate_abort', 'intercase_n_1__W_Assess potential fraud__complete', 'intercase_n_1__W_Assess potential fraud__resume', 'intercase_n_1__W_Assess potential fraud__schedule', 'intercase_n_1__W_Assess potential fraud__start', 'intercase_n_1__W_Assess potential fraud__suspend', 'intercase_n_1__W_Assess potential fraud__withdraw', 'intercase_n_1__W_Call after offers__ate_abort', 'intercase_n_1__W_Call after offers__complete', 'intercase_n_1__W_Call after offers__resume', 'intercase_n_1__W_Call after offers__schedule', 'intercase_n_1__W_Call after offers__start', 'intercase_n_1__W_Call after offers__suspend', 'intercase_n_1__W_Call after offers__withdraw', 'intercase_n_1__W_Call incomplete files__ate_abort', 'intercase_n_1__W_Call incomplete files__complete', 'intercase_n_1__W_Call incomplete files__resume', 'intercase_n_1__W_Call incomplete files__schedule', 'intercase_n_1__W_Call incomplete files__start', 'intercase_n_1__W_Call incomplete files__suspend', 'intercase_n_1__W_Complete application__ate_abort', 'intercase_n_1__W_Complete application__complete', 'intercase_n_1__W_Complete application__resume', 'intercase_n_1__W_Complete application__schedule', 'intercase_n_1__W_Complete application__start', 'intercase_n_1__W_Complete application__suspend', 'intercase_n_1__W_Handle leads__complete', 'intercase_n_1__W_Handle leads__resume', 'intercase_n_1__W_Handle leads__schedule', 'intercase_n_1__W_Handle leads__start', 'intercase_n_1__W_Handle leads__suspend', 'intercase_n_1__W_Handle leads__withdraw', 'intercase_n_1__W_Shortened completion __resume', 'intercase_n_1__W_Shortened completion __schedule', 'intercase_n_1__W_Shortened completion __start', 'intercase_n_1__W_Shortened completion __suspend', 'intercase_n_1__W_Validate application__ate_abort', 'intercase_n_1__W_Validate application__complete', 'intercase_n_1__W_Validate application__resume', 'intercase_n_1__W_Validate application__schedule', 'intercase_n_1__W_Validate application__start', 'intercase_n_1__W_Validate application__suspend']
ii3 = ['intercase_n_3__W_Assess potential fraud__ate_abort_W_Assess potential fraud__schedule_W_Assess potential fraud__start', 'intercase_n_3__W_Assess potential fraud__ate_abort_W_Validate application__schedule_W_Validate application__start', 'intercase_n_3__W_Assess potential fraud__complete_W_Assess potential fraud__schedule_W_Assess potential fraud__start', 'intercase_n_3__W_Assess potential fraud__complete_W_Complete application__schedule_W_Complete application__start', 'intercase_n_3__W_Assess potential fraud__complete_W_Validate application__schedule_W_Validate application__start', 'intercase_n_3__W_Assess potential fraud__resume_W_Assess potential fraud__complete_W_Validate application__schedule', 'intercase_n_3__W_Assess potential fraud__resume_W_Assess potential fraud__start_W_Assess potential fraud__suspend', 'intercase_n_3__W_Assess potential fraud__resume_W_Assess potential fraud__start_W_Call after offers__schedule', 'intercase_n_3__W_Assess potential fraud__resume_W_Assess potential fraud__start_W_Handle leads__schedule', 'intercase_n_3__W_Assess potential fraud__resume_W_Assess potential fraud__start_W_Validate application__schedule', 'intercase_n_3__W_Assess potential fraud__resume_W_Assess potential fraud__suspend_W_Assess potential fraud__resume', 'intercase_n_3__W_Assess potential fraud__resume_W_Assess potential fraud__suspend_W_Complete application__schedule', 'intercase_n_3__W_Assess potential fraud__resume_W_Assess potential fraud__suspend_W_Validate application__schedule', 'intercase_n_3__W_Assess potential fraud__resume_W_Call after offers__schedule_W_Call after offers__start', 'intercase_n_3__W_Assess potential fraud__resume_W_Complete application__schedule_W_Complete application__start', 'intercase_n_3__W_Assess potential fraud__resume_W_Handle leads__schedule_W_Handle leads__start', 'intercase_n_3__W_Assess potential fraud__resume_W_Validate application__schedule_W_Validate application__start', 'intercase_n_3__W_Assess potential fraud__schedule_W_Assess potential fraud__start_W_Assess potential fraud__complete', 'intercase_n_3__W_Assess potential fraud__schedule_W_Assess potential fraud__start_W_Assess potential fraud__suspend', 'intercase_n_3__W_Assess potential fraud__schedule_W_Assess potential fraud__start_W_Call after offers__schedule', 'intercase_n_3__W_Assess potential fraud__schedule_W_Assess potential fraud__start_W_Handle leads__schedule', 'intercase_n_3__W_Assess potential fraud__schedule_W_Assess potential fraud__withdraw_W_Handle leads__schedule', 'intercase_n_3__W_Assess potential fraud__schedule_W_Handle leads__schedule_W_Handle leads__start', 'intercase_n_3__W_Assess potential fraud__start_W_Assess potential fraud__complete_W_Assess potential fraud__schedule', 'intercase_n_3__W_Assess potential fraud__start_W_Assess potential fraud__complete_W_Complete application__schedule', 'intercase_n_3__W_Assess potential fraud__start_W_Assess potential fraud__suspend_W_Assess potential fraud__ate_abort', 'intercase_n_3__W_Assess potential fraud__start_W_Assess potential fraud__suspend_W_Assess potential fraud__resume', 'intercase_n_3__W_Assess potential fraud__start_W_Assess potential fraud__suspend_W_Call after offers__schedule', 'intercase_n_3__W_Assess potential fraud__start_W_Assess potential fraud__suspend_W_Complete application__schedule', 'intercase_n_3__W_Assess potential fraud__start_W_Assess potential fraud__suspend_W_Handle leads__schedule', 'intercase_n_3__W_Assess potential fraud__start_W_Assess potential fraud__suspend_W_Validate application__schedule', 'intercase_n_3__W_Assess potential fraud__start_W_Call after offers__schedule_W_Call after offers__start', 'intercase_n_3__W_Assess potential fraud__start_W_Handle leads__schedule_W_Handle leads__start', 'intercase_n_3__W_Assess potential fraud__start_W_Validate application__schedule_W_Validate application__start', 'intercase_n_3__W_Assess potential fraud__suspend_W_Assess potential fraud__ate_abort_W_Assess potential fraud__schedule', 'intercase_n_3__W_Assess potential fraud__suspend_W_Assess potential fraud__ate_abort_W_Validate application__schedule', 'intercase_n_3__W_Assess potential fraud__suspend_W_Assess potential fraud__resume_W_Assess potential fraud__complete', 'intercase_n_3__W_Assess potential fraud__suspend_W_Assess potential fraud__resume_W_Assess potential fraud__start', 'intercase_n_3__W_Assess potential fraud__suspend_W_Assess potential fraud__resume_W_Assess potential fraud__suspend', 'intercase_n_3__W_Assess potential fraud__suspend_W_Assess potential fraud__resume_W_Call after offers__schedule', 'intercase_n_3__W_Assess potential fraud__suspend_W_Assess potential fraud__resume_W_Complete application__schedule', 'intercase_n_3__W_Assess potential fraud__suspend_W_Assess potential fraud__resume_W_Handle leads__schedule', 'intercase_n_3__W_Assess potential fraud__suspend_W_Assess potential fraud__resume_W_Validate application__schedule', 'intercase_n_3__W_Assess potential fraud__suspend_W_Call after offers__schedule_W_Call after offers__start', 'intercase_n_3__W_Assess potential fraud__suspend_W_Complete application__schedule_W_Complete application__start', 'intercase_n_3__W_Assess potential fraud__suspend_W_Handle leads__schedule_W_Handle leads__start', 'intercase_n_3__W_Assess potential fraud__suspend_W_Validate application__schedule_W_Validate application__start', 'intercase_n_3__W_Assess potential fraud__withdraw_W_Handle leads__schedule_W_Handle leads__start', 'intercase_n_3__W_Call after offers__ate_abort_W_Assess potential fraud__schedule_W_Assess potential fraud__start', 'intercase_n_3__W_Call after offers__ate_abort_W_Call after offers__schedule_W_Assess potential fraud__schedule', 'intercase_n_3__W_Call after offers__ate_abort_W_Call after offers__schedule_W_Call after offers__start', 'intercase_n_3__W_Call after offers__ate_abort_W_Call after offers__schedule_W_Call after offers__withdraw', 'intercase_n_3__W_Call after offers__ate_abort_W_Call after offers__schedule_W_Validate application__schedule', 'intercase_n_3__W_Call after offers__ate_abort_W_Validate application__schedule_W_Validate application__start', 'intercase_n_3__W_Call after offers__complete_W_Validate application__schedule_W_Validate application__start', 'intercase_n_3__W_Call after offers__resume_W_Call after offers__start_W_Call after offers__suspend', 'intercase_n_3__W_Call after offers__resume_W_Call after offers__suspend_W_Assess potential fraud__schedule', 'intercase_n_3__W_Call after offers__resume_W_Call after offers__suspend_W_Call after offers__ate_abort', 'intercase_n_3__W_Call after offers__resume_W_Call after offers__suspend_W_Call after offers__resume', 'intercase_n_3__W_Call after offers__resume_W_Call after offers__suspend_W_Shortened completion __resume', 'intercase_n_3__W_Call after offers__resume_W_Call after offers__suspend_W_Validate application__schedule', 'intercase_n_3__W_Call after offers__resume_W_Validate application__schedule_W_Validate application__start', 'intercase_n_3__W_Call after offers__schedule_W_Assess potential fraud__schedule_W_Assess potential fraud__start', 'intercase_n_3__W_Call after offers__schedule_W_Call after offers__start_W_Call after offers__complete', 'intercase_n_3__W_Call after offers__schedule_W_Call after offers__start_W_Call after offers__suspend', 'intercase_n_3__W_Call after offers__schedule_W_Call after offers__start_W_Shortened completion __schedule', 'intercase_n_3__W_Call after offers__schedule_W_Call after offers__start_W_Shortened completion __suspend', 'intercase_n_3__W_Call after offers__schedule_W_Call after offers__start_W_Validate application__schedule', 'intercase_n_3__W_Call after offers__schedule_W_Call after offers__withdraw_W_Call after offers__schedule', 'intercase_n_3__W_Call after offers__schedule_W_Call after offers__withdraw_W_Validate application__schedule', 'intercase_n_3__W_Call after offers__schedule_W_Validate application__schedule_W_Validate application__start', 'intercase_n_3__W_Call after offers__start_W_Call after offers__complete_W_Validate application__schedule', 'intercase_n_3__W_Call after offers__start_W_Call after offers__suspend_W_Assess potential fraud__schedule', 'intercase_n_3__W_Call after offers__start_W_Call after offers__suspend_W_Call after offers__ate_abort', 'intercase_n_3__W_Call after offers__start_W_Call after offers__suspend_W_Call after offers__resume', 'intercase_n_3__W_Call after offers__start_W_Call after offers__suspend_W_Shortened completion __schedule', 'intercase_n_3__W_Call after offers__start_W_Call after offers__suspend_W_Validate application__schedule', 'intercase_n_3__W_Call after offers__start_W_Shortened completion __schedule_W_Shortened completion __start', 'intercase_n_3__W_Call after offers__start_W_Shortened completion __suspend_W_Call after offers__suspend', 'intercase_n_3__W_Call after offers__start_W_Validate application__schedule_W_Validate application__start', 'intercase_n_3__W_Call after offers__suspend_W_Assess potential fraud__schedule_W_Assess potential fraud__start', 'intercase_n_3__W_Call after offers__suspend_W_Call after offers__ate_abort_W_Assess potential fraud__schedule', 'intercase_n_3__W_Call after offers__suspend_W_Call after offers__ate_abort_W_Call after offers__schedule', 'intercase_n_3__W_Call after offers__suspend_W_Call after offers__ate_abort_W_Validate application__schedule', 'intercase_n_3__W_Call after offers__suspend_W_Call after offers__resume_W_Call after offers__start', 'intercase_n_3__W_Call after offers__suspend_W_Call after offers__resume_W_Call after offers__suspend', 'intercase_n_3__W_Call after offers__suspend_W_Call after offers__resume_W_Validate application__schedule', 'intercase_n_3__W_Call after offers__suspend_W_Shortened completion __schedule_W_Shortened completion __start', 'intercase_n_3__W_Call after offers__suspend_W_Validate application__schedule_W_Validate application__start', 'intercase_n_3__W_Call after offers__withdraw_W_Call after offers__schedule_W_Call after offers__start', 'intercase_n_3__W_Call after offers__withdraw_W_Validate application__schedule_W_Validate application__start', 'intercase_n_3__W_Call incomplete files__ate_abort_W_Assess potential fraud__schedule_W_Assess potential fraud__start', 'intercase_n_3__W_Call incomplete files__ate_abort_W_Call after offers__schedule_W_Call after offers__start', 'intercase_n_3__W_Call incomplete files__ate_abort_W_Call incomplete files__schedule_W_Call incomplete files__start', 'intercase_n_3__W_Call incomplete files__ate_abort_W_Validate application__schedule_W_Validate application__start', 'intercase_n_3__W_Call incomplete files__complete_W_Assess potential fraud__schedule_W_Assess potential fraud__start', 'intercase_n_3__W_Call incomplete files__complete_W_Validate application__schedule_W_Validate application__start', 'intercase_n_3__W_Call incomplete files__resume_W_Assess potential fraud__schedule_W_Assess potential fraud__start', 'intercase_n_3__W_Call incomplete files__resume_W_Call incomplete files__complete_W_Validate application__schedule', 'intercase_n_3__W_Call incomplete files__resume_W_Call incomplete files__start_W_Call incomplete files__complete', 'intercase_n_3__W_Call incomplete files__resume_W_Call incomplete files__start_W_Call incomplete files__suspend', 'intercase_n_3__W_Call incomplete files__resume_W_Call incomplete files__start_W_Validate application__schedule', 'intercase_n_3__W_Call incomplete files__resume_W_Call incomplete files__suspend_W_Assess potential fraud__schedule', 'intercase_n_3__W_Call incomplete files__resume_W_Call incomplete files__suspend_W_Call after offers__schedule', 'intercase_n_3__W_Call incomplete files__resume_W_Call incomplete files__suspend_W_Call incomplete files__ate_abort', 'intercase_n_3__W_Call incomplete files__resume_W_Call incomplete files__suspend_W_Call incomplete files__resume', 'intercase_n_3__W_Call incomplete files__resume_W_Call incomplete files__suspend_W_Validate application__schedule', 'intercase_n_3__W_Call incomplete files__resume_W_Validate application__schedule_W_Validate application__start', 'intercase_n_3__W_Call incomplete files__schedule_W_Call incomplete files__start_W_Assess potential fraud__schedule', 'intercase_n_3__W_Call incomplete files__schedule_W_Call incomplete files__start_W_Call incomplete files__complete', 'intercase_n_3__W_Call incomplete files__schedule_W_Call incomplete files__start_W_Call incomplete files__suspend', 'intercase_n_3__W_Call incomplete files__schedule_W_Call incomplete files__start_W_Validate application__schedule', 'intercase_n_3__W_Call incomplete files__start_W_Assess potential fraud__schedule_W_Assess potential fraud__start', 'intercase_n_3__W_Call incomplete files__start_W_Call incomplete files__complete_W_Assess potential fraud__schedule', 'intercase_n_3__W_Call incomplete files__start_W_Call incomplete files__complete_W_Validate application__schedule', 'intercase_n_3__W_Call incomplete files__start_W_Call incomplete files__suspend_W_Assess potential fraud__schedule', 'intercase_n_3__W_Call incomplete files__start_W_Call incomplete files__suspend_W_Call incomplete files__ate_abort', 'intercase_n_3__W_Call incomplete files__start_W_Call incomplete files__suspend_W_Call incomplete files__resume', 'intercase_n_3__W_Call incomplete files__start_W_Call incomplete files__suspend_W_Validate application__schedule', 'intercase_n_3__W_Call incomplete files__start_W_Validate application__schedule_W_Validate application__start', 'intercase_n_3__W_Call incomplete files__suspend_W_Assess potential fraud__schedule_W_Assess potential fraud__start', 'intercase_n_3__W_Call incomplete files__suspend_W_Call after offers__schedule_W_Call after offers__start', 'intercase_n_3__W_Call incomplete files__suspend_W_Call incomplete files__ate_abort_W_Assess potential fraud__schedule', 'intercase_n_3__W_Call incomplete files__suspend_W_Call incomplete files__ate_abort_W_Call after offers__schedule', 'intercase_n_3__W_Call incomplete files__suspend_W_Call incomplete files__ate_abort_W_Call incomplete files__schedule', 'intercase_n_3__W_Call incomplete files__suspend_W_Call incomplete files__ate_abort_W_Validate application__schedule', 'intercase_n_3__W_Call incomplete files__suspend_W_Call incomplete files__resume_W_Assess potential fraud__schedule', 'intercase_n_3__W_Call incomplete files__suspend_W_Call incomplete files__resume_W_Call incomplete files__complete', 'intercase_n_3__W_Call incomplete files__suspend_W_Call incomplete files__resume_W_Call incomplete files__start', 'intercase_n_3__W_Call incomplete files__suspend_W_Call incomplete files__resume_W_Call incomplete files__suspend', 'intercase_n_3__W_Call incomplete files__suspend_W_Call incomplete files__resume_W_Validate application__schedule', 'intercase_n_3__W_Call incomplete files__suspend_W_Validate application__schedule_W_Validate application__start', 'intercase_n_3__W_Complete application__ate_abort_W_Assess potential fraud__schedule_W_Assess potential fraud__start', 'intercase_n_3__W_Complete application__ate_abort_W_Complete application__schedule_W_Complete application__start', 'intercase_n_3__W_Complete application__complete_W_Assess potential fraud__schedule_W_Assess potential fraud__start', 'intercase_n_3__W_Complete application__complete_W_Complete application__schedule_W_Complete application__start', 'intercase_n_3__W_Complete application__resume_W_Call after offers__schedule_W_Call after offers__start', 'intercase_n_3__W_Complete application__resume_W_Complete application__start_W_Call after offers__schedule', 'intercase_n_3__W_Complete application__resume_W_Complete application__start_W_Complete application__suspend', 'intercase_n_3__W_Complete application__resume_W_Complete application__suspend_W_Call after offers__schedule', 'intercase_n_3__W_Complete application__resume_W_Complete application__suspend_W_Complete application__ate_abort', 'intercase_n_3__W_Complete application__resume_W_Complete application__suspend_W_Complete application__resume', 'intercase_n_3__W_Complete application__resume_W_Shortened completion __schedule_W_Shortened completion __start', 'intercase_n_3__W_Complete application__schedule', 'intercase_n_3__W_Complete application__schedule_W_Call after offers__schedule_W_Call after offers__start', 'intercase_n_3__W_Complete application__schedule_W_Complete application__start', 'intercase_n_3__W_Complete application__schedule_W_Complete application__start_W_Call after offers__schedule', 'intercase_n_3__W_Complete application__schedule_W_Complete application__start_W_Complete application__complete', 'intercase_n_3__W_Complete application__schedule_W_Complete application__start_W_Complete application__suspend', 'intercase_n_3__W_Complete application__schedule_W_Complete application__start_W_Shortened completion __schedule', 'intercase_n_3__W_Complete application__schedule_W_Shortened completion __schedule_W_Shortened completion __start', 'intercase_n_3__W_Complete application__start_W_Call after offers__schedule_W_Call after offers__start', 'intercase_n_3__W_Complete application__start_W_Complete application__complete_W_Assess potential fraud__schedule', 'intercase_n_3__W_Complete application__start_W_Complete application__complete_W_Complete application__schedule', 'intercase_n_3__W_Complete application__start_W_Complete application__suspend_W_Assess potential fraud__schedule', 'intercase_n_3__W_Complete application__start_W_Complete application__suspend_W_Call after offers__schedule', 'intercase_n_3__W_Complete application__start_W_Complete application__suspend_W_Complete application__ate_abort', 'intercase_n_3__W_Complete application__start_W_Complete application__suspend_W_Complete application__resume', 'intercase_n_3__W_Complete application__start_W_Complete application__suspend_W_Shortened completion __schedule', 'intercase_n_3__W_Complete application__start_W_Shortened completion __schedule_W_Shortened completion __start', 'intercase_n_3__W_Complete application__suspend_W_Assess potential fraud__schedule_W_Assess potential fraud__start', 'intercase_n_3__W_Complete application__suspend_W_Call after offers__schedule_W_Call after offers__start', 'intercase_n_3__W_Complete application__suspend_W_Complete application__ate_abort_W_Assess potential fraud__schedule', 'intercase_n_3__W_Complete application__suspend_W_Complete application__ate_abort_W_Complete application__schedule', 'intercase_n_3__W_Complete application__suspend_W_Complete application__resume_W_Call after offers__schedule', 'intercase_n_3__W_Complete application__suspend_W_Complete application__resume_W_Complete application__start', 'intercase_n_3__W_Complete application__suspend_W_Complete application__resume_W_Complete application__suspend', 'intercase_n_3__W_Complete application__suspend_W_Complete application__resume_W_Shortened completion __schedule', 'intercase_n_3__W_Complete application__suspend_W_Shortened completion __schedule_W_Shortened completion __start', 'intercase_n_3__W_Handle leads__complete_W_Handle leads__schedule_W_Handle leads__start', 'intercase_n_3__W_Handle leads__resume_W_Complete application__schedule_W_Call after offers__schedule', 'intercase_n_3__W_Handle leads__resume_W_Complete application__schedule_W_Complete application__start', 'intercase_n_3__W_Handle leads__resume_W_Handle leads__complete_W_Handle leads__schedule', 'intercase_n_3__W_Handle leads__resume_W_Handle leads__start_W_Complete application__schedule', 'intercase_n_3__W_Handle leads__resume_W_Handle leads__start_W_Handle leads__suspend', 'intercase_n_3__W_Handle leads__resume_W_Handle leads__suspend_W_Complete application__schedule', 'intercase_n_3__W_Handle leads__resume_W_Handle leads__suspend_W_Handle leads__resume', 'intercase_n_3__W_Handle leads__schedule', 'intercase_n_3__W_Handle leads__schedule_W_Complete application__schedule', 'intercase_n_3__W_Handle leads__schedule_W_Complete application__schedule_W_Call after offers__schedule', 'intercase_n_3__W_Handle leads__schedule_W_Complete application__schedule_W_Complete application__start', 'intercase_n_3__W_Handle leads__schedule_W_Complete application__schedule_W_Shortened completion __schedule', 'intercase_n_3__W_Handle leads__schedule_W_Handle leads__start', 'intercase_n_3__W_Handle leads__schedule_W_Handle leads__start_W_Complete application__schedule', 'intercase_n_3__W_Handle leads__schedule_W_Handle leads__start_W_Handle leads__suspend', 'intercase_n_3__W_Handle leads__schedule_W_Handle leads__withdraw', 'intercase_n_3__W_Handle leads__schedule_W_Handle leads__withdraw_W_Assess potential fraud__schedule', 'intercase_n_3__W_Handle leads__start_W_Complete application__schedule_W_Call after offers__schedule', 'intercase_n_3__W_Handle leads__start_W_Complete application__schedule_W_Complete application__start', 'intercase_n_3__W_Handle leads__start_W_Handle leads__suspend_W_Complete application__schedule', 'intercase_n_3__W_Handle leads__start_W_Handle leads__suspend_W_Handle leads__resume', 'intercase_n_3__W_Handle leads__suspend_W_Complete application__schedule_W_Call after offers__schedule', 'intercase_n_3__W_Handle leads__suspend_W_Complete application__schedule_W_Complete application__start', 'intercase_n_3__W_Handle leads__suspend_W_Handle leads__resume_W_Complete application__schedule', 'intercase_n_3__W_Handle leads__suspend_W_Handle leads__resume_W_Handle leads__complete', 'intercase_n_3__W_Handle leads__suspend_W_Handle leads__resume_W_Handle leads__start', 'intercase_n_3__W_Handle leads__suspend_W_Handle leads__resume_W_Handle leads__suspend', 'intercase_n_3__W_Handle leads__withdraw_W_Assess potential fraud__schedule_W_Assess potential fraud__start', 'intercase_n_3__W_Handle leads__withdraw_W_Assess potential fraud__schedule_W_Assess potential fraud__withdraw', 'intercase_n_3__W_Handle leads__withdraw_W_Assess potential fraud__schedule_W_Handle leads__schedule', 'intercase_n_3__W_Shortened completion __resume_W_Shortened completion __suspend_W_Shortened completion __resume', 'intercase_n_3__W_Shortened completion __resume_W_Validate application__ate_abort_W_Call incomplete files__schedule', 'intercase_n_3__W_Shortened completion __resume_W_Validate application__resume_W_Validate application__suspend', 'intercase_n_3__W_Shortened completion __schedule_W_Shortened completion __start_W_Call after offers__resume', 'intercase_n_3__W_Shortened completion __schedule_W_Shortened completion __start_W_Call after offers__schedule', 'intercase_n_3__W_Shortened completion __schedule_W_Shortened completion __start_W_Call after offers__suspend', 'intercase_n_3__W_Shortened completion __schedule_W_Shortened completion __start_W_Complete application__suspend', 'intercase_n_3__W_Shortened completion __schedule_W_Shortened completion __start_W_Shortened completion __suspend', 'intercase_n_3__W_Shortened completion __schedule_W_Shortened completion __start_W_Validate application__schedule', 'intercase_n_3__W_Shortened completion __start_W_Call after offers__resume_W_Call after offers__suspend', 'intercase_n_3__W_Shortened completion __start_W_Call after offers__schedule_W_Call after offers__start', 'intercase_n_3__W_Shortened completion __start_W_Call after offers__suspend_W_Call after offers__resume', 'intercase_n_3__W_Shortened completion __start_W_Call after offers__suspend_W_Validate application__schedule', 'intercase_n_3__W_Shortened completion __start_W_Complete application__suspend_W_Call after offers__schedule', 'intercase_n_3__W_Shortened completion __start_W_Complete application__suspend_W_Complete application__resume', 'intercase_n_3__W_Shortened completion __start_W_Shortened completion __suspend_W_Call after offers__resume', 'intercase_n_3__W_Shortened completion __start_W_Shortened completion __suspend_W_Complete application__start', 'intercase_n_3__W_Shortened completion __start_W_Shortened completion __suspend_W_Complete application__suspend', 'intercase_n_3__W_Shortened completion __start_W_Shortened completion __suspend_W_Shortened completion __resume', 'intercase_n_3__W_Shortened completion __start_W_Validate application__schedule_W_Validate application__start', 'intercase_n_3__W_Shortened completion __suspend_W_Call after offers__resume_W_Call after offers__suspend', 'intercase_n_3__W_Shortened completion __suspend_W_Call after offers__suspend_W_Call after offers__resume', 'intercase_n_3__W_Shortened completion __suspend_W_Complete application__start_W_Complete application__suspend', 'intercase_n_3__W_Shortened completion __suspend_W_Complete application__suspend_W_Complete application__resume', 'intercase_n_3__W_Shortened completion __suspend_W_Shortened completion __resume_W_Shortened completion __suspend', 'intercase_n_3__W_Validate application__ate_abort_W_Assess potential fraud__schedule_W_Assess potential fraud__start', 'intercase_n_3__W_Validate application__ate_abort_W_Call incomplete files__schedule_W_Call incomplete files__start', 'intercase_n_3__W_Validate application__ate_abort_W_Validate application__schedule_W_Validate application__start', 'intercase_n_3__W_Validate application__complete_W_Assess potential fraud__schedule_W_Assess potential fraud__start', 'intercase_n_3__W_Validate application__complete_W_Call incomplete files__schedule_W_Call incomplete files__start', 'intercase_n_3__W_Validate application__complete_W_Validate application__schedule_W_Validate application__start', 'intercase_n_3__W_Validate application__resume_W_Assess potential fraud__schedule_W_Assess potential fraud__start', 'intercase_n_3__W_Validate application__resume_W_Call incomplete files__schedule_W_Call incomplete files__start', 'intercase_n_3__W_Validate application__resume_W_Validate application__complete_W_Assess potential fraud__schedule', 'intercase_n_3__W_Validate application__resume_W_Validate application__complete_W_Call incomplete files__schedule', 'intercase_n_3__W_Validate application__resume_W_Validate application__start_W_Call incomplete files__schedule', 'intercase_n_3__W_Validate application__resume_W_Validate application__start_W_Validate application__complete', 'intercase_n_3__W_Validate application__resume_W_Validate application__start_W_Validate application__suspend', 'intercase_n_3__W_Validate application__resume_W_Validate application__suspend_W_Assess potential fraud__schedule', 'intercase_n_3__W_Validate application__resume_W_Validate application__suspend_W_Call incomplete files__schedule', 'intercase_n_3__W_Validate application__resume_W_Validate application__suspend_W_Shortened completion __resume', 'intercase_n_3__W_Validate application__resume_W_Validate application__suspend_W_Validate application__ate_abort', 'intercase_n_3__W_Validate application__resume_W_Validate application__suspend_W_Validate application__resume', 'intercase_n_3__W_Validate application__schedule_W_Validate application__start_W_Assess potential fraud__schedule', 'intercase_n_3__W_Validate application__schedule_W_Validate application__start_W_Call incomplete files__schedule', 'intercase_n_3__W_Validate application__schedule_W_Validate application__start_W_Validate application__complete', 'intercase_n_3__W_Validate application__schedule_W_Validate application__start_W_Validate application__suspend', 'intercase_n_3__W_Validate application__start_W_Assess potential fraud__schedule_W_Assess potential fraud__start', 'intercase_n_3__W_Validate application__start_W_Call incomplete files__schedule_W_Call incomplete files__start', 'intercase_n_3__W_Validate application__start_W_Validate application__complete_W_Assess potential fraud__schedule', 'intercase_n_3__W_Validate application__start_W_Validate application__complete_W_Call incomplete files__schedule', 'intercase_n_3__W_Validate application__start_W_Validate application__complete_W_Validate application__schedule', 'intercase_n_3__W_Validate application__start_W_Validate application__suspend_W_Assess potential fraud__schedule', 'intercase_n_3__W_Validate application__start_W_Validate application__suspend_W_Call incomplete files__schedule', 'intercase_n_3__W_Validate application__start_W_Validate application__suspend_W_Shortened completion __resume', 'intercase_n_3__W_Validate application__start_W_Validate application__suspend_W_Validate application__ate_abort', 'intercase_n_3__W_Validate application__start_W_Validate application__suspend_W_Validate application__resume', 'intercase_n_3__W_Validate application__suspend_W_Assess potential fraud__schedule_W_Assess potential fraud__start', 'intercase_n_3__W_Validate application__suspend_W_Call incomplete files__schedule_W_Call incomplete files__start', 'intercase_n_3__W_Validate application__suspend_W_Shortened completion __resume_W_Validate application__ate_abort', 'intercase_n_3__W_Validate application__suspend_W_Shortened completion __resume_W_Validate application__resume', 'intercase_n_3__W_Validate application__suspend_W_Validate application__ate_abort_W_Assess potential fraud__schedule', 'intercase_n_3__W_Validate application__suspend_W_Validate application__ate_abort_W_Call incomplete files__schedule', 'intercase_n_3__W_Validate application__suspend_W_Validate application__ate_abort_W_Validate application__schedule', 'intercase_n_3__W_Validate application__suspend_W_Validate application__resume_W_Assess potential fraud__schedule', 'intercase_n_3__W_Validate application__suspend_W_Validate application__resume_W_Call incomplete files__schedule', 'intercase_n_3__W_Validate application__suspend_W_Validate application__resume_W_Validate application__complete', 'intercase_n_3__W_Validate application__suspend_W_Validate application__resume_W_Validate application__start', 'intercase_n_3__W_Validate application__suspend_W_Validate application__resume_W_Validate application__suspend']

In [3]:
print(list(test_event_log.columns))

['Action_start', 'org:resource_start', 'concept:name', 'EventOrigin_start', 'EventID_start', 'lifecycle:transition_start', 'time:timestamp_start', 'case:LoanGoal_start', 'case:ApplicationType_start', 'case:concept:name', 'case:RequestedAmount_start', 'FirstWithdrawalAmount_start', 'NumberOfTerms_start', 'Accepted_start', 'MonthlyCost_start', 'Selected_start', 'CreditScore_start', 'OfferedAmount_start', 'OfferID_start', 'Action_complete', 'org:resource_complete', 'EventOrigin_complete', 'EventID_complete', 'lifecycle:transition_complete', 'time:timestamp_complete', 'case:LoanGoal_complete', 'case:ApplicationType_complete', 'case:RequestedAmount_complete', 'FirstWithdrawalAmount_complete', 'NumberOfTerms_complete', 'Accepted_complete', 'MonthlyCost_complete', 'Selected_complete', 'CreditScore_complete', 'OfferedAmount_complete', 'OfferID_complete', 'duration', 'duration_seconds', 'duration_ms', 'duration_hours', 'seconds_in_day', 'day_of_week', 'User_1', 'User_10', 'User_100', 'User_101'

In [4]:
N = 1000
import conduct_evaluation
get_pscores = lambda likelihoods : [pscore(likelihoods[1][i], likelihoods[2][k][3]).compute()[0] for i, k in enumerate(list(likelihoods[0].keys()))]

In [5]:
drbart_model= DRBART(parser_dir = '../../../models/standard_numerical_stable/'+model_name+'/crsdar_ii1/',
                     strict_parser=False)
evaluator = conduct_evaluation.ConductEvaluation(drbart_model, SampleOutcomes_DRBART_Normal_A_R_S_D_AC_RC_II,
                                                   {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource_start',
                                                        'known_resources' : known_resources,
                                                        'known_activities' : known_activities,
                                                        'inter_instance_column_names' : ii1
                                                    },
                                     test_event_log, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods = evaluator.sample_cases(False, True)

  0%|                                                                                                                                                                                                        | 0/6068 [00:00<?, ?it/s]

  1%|█▌                                                                                                                                                                                             | 50/6068 [00:01<02:47, 35.91it/s]

  2%|███▏                                                                                                                                                                                          | 100/6068 [00:01<01:44, 56.84it/s]

  2%|████▋                                                                                                                                                                                         | 150/6068 [00:02<01:13, 80.08it/s]

  3%|██████▎                                                                                                                                                                                       | 200/6068 [00:02<01:00, 97.68it/s]

...

 77%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████                                           | 4676/6068 [00:10<00:00, 1494.65it/s]

 82%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████                                 | 5001/6068 [00:10<00:00, 1553.22it/s]

 94%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▋           | 5701/6068 [00:10<00:00, 2275.91it/s]

 99%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▉  | 6001/6068 [00:10<00:00, 1782.22it/s]

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6068/6068 [00:10<00:00, 566.90it/s]

In [6]:
np.mean([v.ln() for v in likelihoods[0].values()])

Decimal('-6.510380645990667810310698039')

In [7]:
np.mean(get_pscores(likelihoods))

np.float64(1374379.9193082664)